# This is an interactive script to compare the training results of different models. It will load the training logs and output a markdown file with the comparison results.


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_dataset
from difflib import SequenceMatcher
import json, time
from pathlib import Path

In [12]:
import os
print(os.getcwd())

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)


/Users/miles/p8/dev/ai/experiments/fine_tuning
/Users/miles/p8/dev/ai/experiments


In [13]:
# ----------- CONFIG -------------
base_model_name = "Qwen/Qwen2.5-3B-Instruct"
adapter_path = "_outputs/lora_model"
dataset_file = "training_data.json"  # your saved training dataset
max_new_tokens = 2000
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
outputs_dir = "./_outputs/compare_training"
results_dir = "./_results/compare_training"
# --------------------------------


In [14]:
# Load dataset
dataset = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    "zh",
    split="train[0:]",
) #
print(f"• Loaded {len(dataset)} samples.")
print(dataset[0])



• Loaded 20171 samples.
{'Question': '根据描述，一个1岁的孩子在夏季头皮出现多处小结节，长期不愈合，且现在疮大如梅，溃破流脓，口不收敛，头皮下有空洞，患处皮肤增厚。这种病症在中医中诊断为什么病？', 'Complex_CoT': '这个小孩子在夏天头皮上长了些小结节，一直都没好，后来变成了脓包，流了好多脓。想想夏天那么热，可能和湿热有关。才一岁的小孩，免疫力本来就不强，夏天的湿热没准就侵袭了身体。\n\n用中医的角度来看，出现小结节、再加上长期不愈合，这些症状让我想到了头疮。小孩子最容易得这些皮肤病，主要因为湿热在体表郁结。\n\n但再看看，头皮下还有空洞，这可能不止是简单的头疮。看起来病情挺严重的，也许是脓肿没治好。这样的情况中医中有时候叫做禿疮或者湿疮，也可能是另一种情况。\n\n等一下，头皮上的空洞和皮肤增厚更像是疾病已经深入到头皮下，这是不是说明有可能是流注或瘰疬？这些名字常描述头部或颈部的严重感染，特别是有化脓不愈合，又形成通道或空洞的情况。\n\n仔细想想，我怎么感觉这些症状更贴近瘰疬的表现？尤其考虑到孩子的年纪和夏天发生的季节性因素，湿热可能是主因，但可能也有火毒或者痰湿造成的滞留。\n\n回到基本的症状描述上看，这种长期不愈合又复杂的状况，如果结合中医更偏重的病名，是不是有可能是涉及更深层次的感染？\n\n再考虑一下，这应该不是单纯的瘰疬，得仔细分析头皮增厚并出现空洞这样的严重症状。中医里头，这样的表现可能更符合‘蚀疮’或‘头疽’。这些病名通常描述头部严重感染后的溃烂和组织坏死。\n\n看看季节和孩子的体质，夏天又湿又热，外邪很容易侵入头部，对孩子这么弱的免疫系统简直就是挑战。头疽这个病名听起来真是切合，因为它描述的感染严重，溃烂到出现空洞。\n\n不过，仔细琢磨后发现，还有个病名似乎更为合适，叫做‘蝼蛄疖’，这病在中医里专指像这种严重感染并伴有深部空洞的情况。它也涵盖了化脓和皮肤增厚这些症状。\n\n哦，该不会是夏季湿热，导致湿毒入侵，孩子的体质不能御，其病情发展成这样的感染？综合分析后我觉得‘蝼蛄疖’这个病名真是相当符合。', 'Response': '从中医的角度来看，你所描述的症状符合“蝼蛄疖”的病症。这种病症通常发生在头皮，表现为多处结节，溃破流脓，形成空洞，患处皮肤增厚且长期不愈合。湿热较重的夏季更容

In [15]:
# Print some samples to verify
print(f"Question: {dataset[0]['Question']}")
print(f"Complex_CoT: {dataset[0]['Complex_CoT']}")
print(f"Response: {dataset[0]['Response']}")

print(f"Question: {dataset[1]['Question']}")
print(f"Complex_CoT: {dataset[1]['Complex_CoT']}")
print(f"Response: {dataset[1]['Response']}")

r_dir = Path(results_dir)
r_dir.mkdir(parents=True, exist_ok=True)
f = open(f"{results_dir}/dataset.txt", "w", encoding="utf-8")



Question: 根据描述，一个1岁的孩子在夏季头皮出现多处小结节，长期不愈合，且现在疮大如梅，溃破流脓，口不收敛，头皮下有空洞，患处皮肤增厚。这种病症在中医中诊断为什么病？
Complex_CoT: 这个小孩子在夏天头皮上长了些小结节，一直都没好，后来变成了脓包，流了好多脓。想想夏天那么热，可能和湿热有关。才一岁的小孩，免疫力本来就不强，夏天的湿热没准就侵袭了身体。

用中医的角度来看，出现小结节、再加上长期不愈合，这些症状让我想到了头疮。小孩子最容易得这些皮肤病，主要因为湿热在体表郁结。

但再看看，头皮下还有空洞，这可能不止是简单的头疮。看起来病情挺严重的，也许是脓肿没治好。这样的情况中医中有时候叫做禿疮或者湿疮，也可能是另一种情况。

等一下，头皮上的空洞和皮肤增厚更像是疾病已经深入到头皮下，这是不是说明有可能是流注或瘰疬？这些名字常描述头部或颈部的严重感染，特别是有化脓不愈合，又形成通道或空洞的情况。

仔细想想，我怎么感觉这些症状更贴近瘰疬的表现？尤其考虑到孩子的年纪和夏天发生的季节性因素，湿热可能是主因，但可能也有火毒或者痰湿造成的滞留。

回到基本的症状描述上看，这种长期不愈合又复杂的状况，如果结合中医更偏重的病名，是不是有可能是涉及更深层次的感染？

再考虑一下，这应该不是单纯的瘰疬，得仔细分析头皮增厚并出现空洞这样的严重症状。中医里头，这样的表现可能更符合‘蚀疮’或‘头疽’。这些病名通常描述头部严重感染后的溃烂和组织坏死。

看看季节和孩子的体质，夏天又湿又热，外邪很容易侵入头部，对孩子这么弱的免疫系统简直就是挑战。头疽这个病名听起来真是切合，因为它描述的感染严重，溃烂到出现空洞。

不过，仔细琢磨后发现，还有个病名似乎更为合适，叫做‘蝼蛄疖’，这病在中医里专指像这种严重感染并伴有深部空洞的情况。它也涵盖了化脓和皮肤增厚这些症状。

哦，该不会是夏季湿热，导致湿毒入侵，孩子的体质不能御，其病情发展成这样的感染？综合分析后我觉得‘蝼蛄疖’这个病名真是相当符合。
Response: 从中医的角度来看，你所描述的症状符合“蝼蛄疖”的病症。这种病症通常发生在头皮，表现为多处结节，溃破流脓，形成空洞，患处皮肤增厚且长期不愈合。湿热较重的夏季更容易导致这种病症的发展，特别是在免疫力较弱的儿童身上。建议结合中医的清热解毒、祛湿消肿的治疗方法进行处理，并配合专

In [16]:
# Set cwd to the nearest parent that contains a ".venv_lora" directory (search PROJECT_ROOT then notebook cwd)
target = None
candidates = [PROJECT_ROOT, Path.cwd()]
seen = set()

for start in candidates:
    for p in [start] + list(start.parents):
        p = p.resolve()
        if p in seen:
            continue
        seen.add(p)
        if (p / ".venv_lora").exists():
            target = p
            break
    if target:
        break

if target:
    os.chdir(str(target))
    print("Changed cwd to:", Path.cwd())
else:
    print("'.venv_lora' not found; cwd unchanged:", Path.cwd())

Changed cwd to: /Users/miles/p8/dev/ai/experiments/fine_tuning


In [17]:


# Load model & tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_path)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, adapter_path)

# base_model.compile()  # compile the base model for faster inference (PyTorch 2.0+)
base_model.eval()
# model.compile()  # compile the model for faster inference (PyTorch 2.0+)
model.eval()
# Move models to the correct device
base_model.to(device)
model.to(device)


Loading weights: 100%|██████████| 434/434 [00:02<00:00, 216.95it/s, Materializing param=model.norm.weight]                              


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_fe

In [18]:
# open the outputresult file
r_dir = Path(results_dir)
r_dir.mkdir(parents=True, exist_ok=True)
f = open(f"{results_dir}/comparison.md", "w", encoding="utf-8")


In [19]:
import re

def split_sections(text):
    headers = ["Instruction", "Question", "Response"]
    pattern = re.compile(r'(?im)^(?:#+\s*)?(Instruction|Question|Response)\s*:\s*', re.MULTILINE)
    matches = list(pattern.finditer(text))
    sections = {h: "" for h in headers}
    if matches:
        for i, m in enumerate(matches):
            key = m.group(1).capitalize()
            start = m.end()
            end = matches[i+1].start() if i+1 < len(matches) else len(text)
            sections[key] = text[start:end].strip()
    else:
        # fallback: inline labels
        for h in headers:
            m = re.search(rf'{h}\s*:\s*(.*?)(?=\n[A-Z][a-zA-Z_ ]*?:|\Z)', text, re.S)
            if m:
                sections[h] = m.group(1).strip()
    return sections



In [ ]:
try:
    start = time.perf_counter()
    # Iterate over samples
    for idx, sample in enumerate(dataset):
        prompt = f"### Instruction:\nYou are a medical expert.\n### Question:\n{sample['Question']}\n### Response:\n<think>"
        
        # Tokenize and move to device
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        outputs_base = base_model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens
        )
        generated_base = tokenizer.decode(outputs_base[0], skip_special_tokens=True)

        sections = split_sections(generated_base)
        Instruction = sections["Instruction"]
        Question = sections["Question"]
        Response = sections["Response"]

        f.write(f"# Response from base model ({idx+1}) ==========\n")
        f.write(generated_base)

        # Generate response
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens
        )
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if idx%100==0:
            print(f"Batch @{idx+1} samples in {time.perf_counter() - start:.3f} seconds")
            start = time.perf_counter() 

        f.write(f"\n## Response from trained model ({idx+1}) ----------\n")
        f.write(generated)
        # Also output the training data
        f.write(f"\n## Training data {idx+1} -------------\n")
        f.write(f"{sample['Response']}")
        f.write(f"\n-- End of ({idx+1}) \n\n")
except KeyboardInterrupt as e:
    print(f"Interrupted at sample {idx+1}")
finally:    # Ensure file is closed even if interrupted
    f.close()
    
print("Done! Results saved to outputs/compare_training.md")
